# Lab 1 — Regression

Starter notebook. Implement the linked GitHub issues; no experiments have been run.


## 1. Problem definition and dataset source

TODO: implement and document this section.


In [1]:
%pip install -q skorch imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


## 2. Environment, seeds, and device

TODO: implement and document this section.


In [2]:
import random
import platform
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Python:", platform.python_version())
print("Device:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

for package in [
    "torch", "scikit-learn", "skorch", "pandas",
    "numpy", "matplotlib", "seaborn", "imbalanced-learn"
]:
    print(f"{package}: {version(package)}")

Python: 3.13.15
Device: cuda
GPU: Tesla T4
torch: 2.11.0+cu128
scikit-learn: 1.6.1
skorch: 1.4.0
pandas: 2.3.3
numpy: 2.1.3
matplotlib: 3.10.0
seaborn: 0.13.2
imbalanced-learn: 0.14.2


In [3]:
from torch import nn
from skorch import NeuralNetRegressor

from sklearn.datasets import make_regression
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Small synthetic regression problem for environment verification.
X_smoke, y_smoke = make_regression(
    n_samples=240,
    n_features=6,
    noise=0.1,
    random_state=SEED,
)

X_smoke = X_smoke.astype(np.float32)
y_smoke = y_smoke.astype(np.float32).reshape(-1, 1)
y_smoke /= y_smoke.std()


class SmokeMLP(nn.Module):
    def __init__(self, input_dim=6, hidden_dim=16):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, X):
        return self.network(X)


net = NeuralNetRegressor(
    module=SmokeMLP,
    criterion=nn.MSELoss,
    optimizer=torch.optim.Adam,
    lr=0.01,
    max_epochs=20,
    batch_size=32,
    train_split=False,
    iterator_train__shuffle=True,
    device=DEVICE,
    verbose=0,
)

pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("net", net),
])

search = GridSearchCV(
    estimator=pipeline,
    param_grid={"net__module__hidden_dim": [8, 16]},
    scoring="neg_mean_squared_error",
    cv=KFold(n_splits=2, shuffle=True, random_state=SEED),
    n_jobs=1,
    error_score="raise",
    refit=True,
)

search.fit(X_smoke, y_smoke)

fitted_net = search.best_estimator_.named_steps["net"]
losses = np.array(fitted_net.history[:, "train_loss"])
predictions = search.predict(X_smoke[:5])

assert np.isfinite(losses).all(), "Non-finite training loss."
assert predictions.shape == (5, 1), "Unexpected prediction shape."
assert np.isfinite(predictions).all(), "Non-finite predictions."

print("PASS: PyTorch training, GridSearchCV, and prediction.")
print("Model device:", next(fitted_net.module_.parameters()).device)
print("Best parameters:", search.best_params_)
print("Best CV MSE:", -search.best_score_)
print(f"Refit training loss: {losses[0]:.4f} → {losses[-1]:.4f}")

PASS: PyTorch training, GridSearchCV, and prediction.
Model device: cuda:0
Best parameters: {'net__module__hidden_dim': 16}
Best CV MSE: 0.008058980572968721
Refit training loss: 1.0699 → 0.0026


## 3. Load and inspect NYSE data

TODO: implement and document this section.


In [4]:
DATA_PATH = Path(
    "/kaggle/input/datasets/dgawlik/nyse/prices-split-adjusted.csv"
)

assert DATA_PATH.is_file(), f"Dataset not found: {DATA_PATH}"

prices = pd.read_csv(DATA_PATH)

required_columns = {
    "date", "symbol", "open", "close", "low", "high", "volume"
}
missing_columns = required_columns - set(prices.columns)

assert not missing_columns, f"Missing columns: {missing_columns}"

print("Shape:", prices.shape)
print("Number of stocks:", prices["symbol"].nunique())
print("Date range:", prices["date"].min(), "→", prices["date"].max())

display(prices.head())

Shape: (851264, 7)
Number of stocks: 501
Date range: 2010-01-04 → 2016-12-30


,date,symbol,open,close,low,high,volume
0,2016-01-05,WLTW,123.430000,125.839996,122.309998,126.250000,2163600.0
1,2016-01-06,WLTW,125.239998,119.980003,119.940002,125.540001,2386400.0
2,2016-01-07,WLTW,116.379997,114.949997,114.930000,119.739998,2489500.0
3,2016-01-08,WLTW,115.480003,116.620003,113.500000,117.440002,2006300.0
4,2016-01-11,WLTW,117.010002,114.970001,114.089996,117.330002,1408600.0


## 4. Chronological splits and exploratory analysis

TODO: implement and document this section.


## 5. Training-only preprocessing and persistence baseline

TODO: implement and document this section.


## 6. Baseline PyTorch MLP

TODO: implement and document this section.


## 7. GridSearchCV with skorch and TimeSeriesSplit

TODO: implement and document this section.


## 8. Controlled regularization comparisons

TODO: implement and document this section.


## 9. Frozen-model train/test curves and final metrics

TODO: implement and document this section.


## 10. Interpretation and exported results

TODO: implement and document this section.
